# Create list mates
Author: Ilyas

## 1) import, read data

In [1]:
import geopandas as gpd, pandas as pd
path_geo_area = r"F:\projects\Forest-Hackathon\data\raw\5. Объединение фрагментированных выделов\Выделы под лесосеками.geojson"
path_save_mates = r"F:\projects\Forest-Hackathon\data\processed\mates.csv"
max_count = 10000
df_geo_area = gpd.read_file(path_geo_area)

## 2) select small data

In [2]:
print(f"Всего выделов: {df_geo_area.size}")
df_geo_small_area = df_geo_area.head(max_count)
print(f"Выбранно выделов: {df_geo_small_area.size}")

Всего выделов: 526268
Выбранно выделов: 20000


In [3]:
from ipyleaflet import Map, GeoData, LayersControl, basemaps

xmin, ymin, xmax, ymax = df_geo_small_area.total_bounds
center_lat = (ymin + ymax) / 2
center_lon = (xmin + xmax) / 2

# --- Карта ---
m = Map(
    center=(center_lat, center_lon),
    zoom=6
)

# --- Слой 1: Выделы (голубые) ---
geo_area = GeoData(
    geo_dataframe=df_geo_small_area,
    style={
        'color': 'blue',
        'fillColor': 'lightblue',
        'opacity': 0.5,
        'weight': 1,
        'fillOpacity': 0.4
    },
    hover_style={'fillColor': 'yellow', 'fillOpacity': 0.7},
    name='Выделы'
)
m.add_layer(geo_area)

# --- Контроль слоёв (включать/выключать) ---
m.add_control(LayersControl())

# --- Показать карту ---
m

Map(center=[np.float64(32.37647490404957), np.float64(73.74547245230717)], controls=(ZoomControl(options=['pos…

## 3) Create mates

In [4]:
# Оставляем только нужные данные
df = df_geo_small_area[['registration_no', 'geometry']].copy()
df = df[df.geometry.notna() & ~df.geometry.is_empty].reset_index(drop=True)

# Пространственный индекс
sindex = df.sindex

mates = []

# Ищем соседей каждого полигона
for i, geom in enumerate(df.geometry):
    # Кандидаты, чьи bounding box пересекаются с текущим полигоном
    candidates = sindex.query(geom, predicate="intersects")

    for j in candidates:
        j = int(j)

        # Убираем сам объект и дубликаты пар
        if j <= i:
            continue

        other = df.geometry.iloc[j]

        # Соседство: есть общая граница, но нет пересечения внутренних частей
        if geom.touches(other):
            mates.append((
                df['registration_no'].iloc[i],
                df['registration_no'].iloc[j]
            ))

# Результирующий DataFrame
df_mates = pd.DataFrame(mates, columns=['Выдел1', 'Выдел2'])

print(df_mates.head())
print("Всего пар соседей:", len(df_mates))

         Выдел1        Выдел2
0  18:5:3:23:19  18:5:3:23:22
1  18:5:3:23:19  18:5:3:23:20
2  18:5:3:23:20  18:5:3:23:22
3  18:5:3:23:20  18:5:3:23:21
4  18:5:3:23:21  18:5:3:23:22
Всего пар соседей: 18904


In [5]:
pair = df_mates.sample(n=1).iloc[0]

df_one_mates = df_geo_area[
    df_geo_area['registration_no'].isin([pair['Выдел1'], pair['Выдел2']])
].copy().reset_index(drop=True)

xmin, ymin, xmax, ymax = df_one_mates.total_bounds
center_lat = (ymin + ymax) / 2
center_lon = (xmin + xmax) / 2

# --- Карта ---
m = Map(
    center=(center_lat, center_lon),
    zoom=15
)

# --- Слой 1: Выделы (голубые) ---
geo_area = GeoData(
    geo_dataframe=df_one_mates,
    style={
        'color': 'blue',
        'fillColor': 'lightblue',
        'opacity': 0.5,
        'weight': 1,
        'fillOpacity': 0.4
    },
    hover_style={'fillColor': 'yellow', 'fillOpacity': 0.7},
    name='Выделы'
)
m.add_layer(geo_area)

# --- Контроль слоёв (включать/выключать) ---
m.add_control(LayersControl())

# --- Показать карту ---
m

Map(center=[np.float64(58.18621432658561), np.float64(28.448008508958374)], controls=(ZoomControl(options=['po…

## 4) Save mates

In [49]:
df_mates.to_csv(path_save_mates, index=False)
print(r"Сохранено в {path_save_mates}")

Сохранено в {path_save_mates}
